# Text as Data

**Computational Text Analysis Pack · Module 00 — Text as Data**

*Technique:* Turning a corpus into a table: document, token, type

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Sagaustus/adh-dh-notebooks/blob/main/00-foundations/00_text_as_data.ipynb)

Data: `masakhanews.csv`, `colonial_texts.csv` — from the `data/` folder of this pack.

---

In [ ]:
# Setup — run this first.
import os, warnings
warnings.filterwarnings('ignore')
import numpy as np
import pandas as pd

# The data this notebook needs. It is downloaded from CourseHub automatically;
# uploading by hand is only a fallback if that server is unreachable.
NEEDED = ['masakhanews.csv', 'colonial_texts.csv']
DATA_URL = 'https://coursehub.africandigitalhumanities.ca/ide/pack-data/text/'

def _missing():
    return [f for f in NEEDED if not os.path.exists(f)]

missing = _missing()
if missing:
    # Fetch from CourseHub first, so nobody has to upload anything.
    import urllib.request, urllib.parse
    for _f in list(missing):
        try:
            urllib.request.urlretrieve(DATA_URL + urllib.parse.quote(_f), _f)
            print(f'downloaded {_f}')
        except Exception as _e:
            print(f'could not fetch {_f}: {_e}')
    missing = _missing()

if missing:
    try:
        from google.colab import files
    except ImportError:
        raise SystemExit('Place these next to the notebook: ' + ', '.join(missing))
    # Ask again until everything has arrived. The upload widget returns as soon
    # as you close it, so picking only some of the files would otherwise fail a
    # few lines below with a confusing FileNotFoundError.
    for _ in range(4):
        print('Select ALL of these at once (ctrl-click / cmd-click to multi-select):')
        print('   ' + ', '.join(missing))
        files.upload()
        missing = _missing()
        if not missing:
            break
        print('Still needed: ' + ', '.join(missing))
    if missing:
        raise SystemExit(
            'Missing: ' + ', '.join(missing) + '. Re-run this cell and select '
            'every file listed, or upload them with the folder icon on the left.')

df       = pd.read_csv('masakhanews.csv')
df_colonial = pd.read_csv('colonial_texts.csv')
NIGERIAN = ['Hausa', 'Igbo', 'Yoruba', 'Nigerian Pidgin']
# Most of this pack runs on scikit-learn and the standard library. Also
# available: nltk, spacy, gensim and networkx (modules 14-19), Keras
# (case studies), and a GPU pod reached through /ide/api/neural/ for the
# neural modules — never called directly, so no token lives in a notebook.

print(f'Loaded. df has {len(df):,} rows and {df.shape[1]} columns.')

## What this notebook does

You already analyse text. You read closely, you notice a word recurring, you
argue from patterns you have seen across a body of writing.

This pack does the same work with counting, on more text than anyone can read.
It does not replace reading — it tells you *where to read*. Everything here
starts from one move: **turning a corpus into a table.**

The data is real African-language journalism and real African-archive text.
Nothing in this pack is in copyright.

## The tooling

**pandas** holds the corpus as a table, one row per document — the shape every
method in this course assumes. **`re`** (the standard library's regular
expressions) does the tokenising, written out rather than imported so the
decisions stay visible.

## Three words used precisely

| Word | Means | In this corpus |
|---|---|---|
| **Document** | One unit of text you treat as a whole | One news article |
| **Token** | One occurrence of a word | Every time "ilú" appears |
| **Type** | One distinct word | "ilú" counted once, however often it occurs |

A document of 500 tokens might have only 300 types, because words repeat. That
ratio is a measurable property of style — notebook 02c uses it.

In [ ]:
print(f"{len(df):,} documents in {df['language'].nunique()} languages")
print()
print(f"{'language':<18}{'docs':>7}{'median words':>14}")
print("-" * 39)
for lang, g in df.groupby("language"):
    print(f"{lang:<18}{len(g):>7,}{g['n_words'].median():>14,.0f}")
print()
print(f"Nigerian-language documents: {df['is_nigerian'].sum():,} of {len(df):,}")

## One row is one document

That is the whole trick. Once text is a table, everything you already know about
tables applies: filter, group, count, compare.

The columns are metadata about each document; the `text` column holds the
document itself.

In [ ]:
print("columns:", list(df.columns))
print()
row = df.iloc[0]
for c in ("language", "category", "headline", "n_words"):
    print(f"  {c:<12} {row[c]}")
print()
print("  text (first 300 characters):")
print("   ", row["text"][:300].replace("\n", " "), "…")

In [ ]:
import re

def tokenise(text):
    """Words, lowercased, Unicode-aware.

    `\\w` matches Unicode letters, so ọ, à, ƙ, ɗ and ɓ survive. It does not
    match combining marks, and a tone mark on ọ or ẹ is always one, so the
    pattern adds the combining-accent range U+0300-U+036F; without it ọ̀rọ̀
    splits into ọ + rọ. The common shortcut `[a-z]+` throws away far more and
    quietly changes every count in this notebook — see notebook 03b.
    """
    return re.findall(r"[\w\u0300-\u036f]+", str(text).lower())

sample = "Àwọn ọmọ ìlú ti ṣe ìpàdé lórí ọ̀rọ̀ ẹ̀kọ́ — ƙungiyar ɗalibai ta yi taro"
print("sample     :", sample)
print("tokens     :", tokenise(sample))
print("bare \\w+   :", re.findall(r"\w+", sample.lower()))
print("naive [a-z]:", re.findall(r"[a-z]+", sample.lower()))
print()
print("The naive pattern splits words at every accented letter; bare \\w+")
print("splits them wherever a tone mark sits on ọ or ẹ.")

That last cell is the most important in the pack, and the reason it is written
out rather than imported from a library.

`\w` is Unicode-aware, so Hausa hooked letters and most Yoruba accents survive.
It misses one thing: a tone mark on ọ or ẹ is stored as a separate combining
character, which `\w` does not match, so the pattern adds that range
(`\u0300-\u036f`). Without it, *ọ̀rọ̀* would split into *ọ* and *rọ*.
The shortcut `[a-z]+` — which appears in a great deal of published code — throws
away far more, silently. Notebook 03b measures exactly how much damage that does.

In [ ]:
# Tokens and types for one document.
doc = df.iloc[0]["text"]
tokens = tokenise(doc)
types = set(tokens)
print(f"tokens : {len(tokens):,}")
print(f"types  : {len(types):,}")
print(f"ratio  : {len(types) / len(tokens):.3f}   (1.0 = no word ever repeats)")
print()
print("the ten commonest words in this one document:")
from collections import Counter
for w, n in Counter(tokens).most_common(10):
    print(f"   {w:<16}{n}")

### 🔧 Your turn 1

Change `df.iloc[0]` to a document in a different language and re-run.

Do the commonest words look like function words — the equivalents of *the*, *of*,
*and*? Almost every language's frequency list is dominated by them. That is why
notebook 02b treats them separately.

## The second corpus

`colonial_texts.csv` is a different shape: one row is a *passage* from a book,
not a whole document. The unit of analysis is a decision you make, not a
property of the data.

In [ ]:
print(f"{len(df_colonial):,} passages from {df_colonial['work_id'].nunique()} works")
print()
print(df_colonial[["title", "author", "passage_no", "n_words"]].head(5).to_string(index=False))
print()
print("Note what one row is here: a passage. Compare that to masakhanews,")
print("where one row is a whole article. Neither is more correct — but a")
print("statistic computed across them means different things.")

### 🔧 Your turn 2

Count the documents per language in `df`, and the passages per work in
`df_colonial`.

Which corpus would you use to ask "how do Nigerian newspapers write about
health?", and which to ask "how did Victorian travel writers describe Lagos?"
Say why in one sentence each.

<details markdown="1">
<summary><b>What you should have seen</b> — click to expand</summary>

**Your turn 1.** Yes — every language's top ten is dominated by grammatical
words: *ni, ti, tó, àwọn* in Yoruba, *da, na, a, ya* in Hausa. This is Zipf's law
in action (notebook 02b). It also explains why raw frequency alone rarely
answers a research question: the commonest words are the least distinctive. What
you usually want is notebook 04a's keyness — words that are common *here* and
not elsewhere.

**Your turn 2.** For Nigerian health journalism, `masakhanews` filtered to
`category == "health"` and the four Nigerian languages: the documents are
journalism, contemporary, and labelled by topic. For Victorian descriptions of
Lagos, `colonial_texts` — but with the caveat that it is a keyword-built
collection of writing *about* Africa, largely by Europeans, so it evidences
colonial discourse rather than African writing. Getting that distinction right
is the difference between a finding and a category error.

</details>

## Where this stops

You can turn a corpus into a table and count what is in it. You cannot yet say
whether a count is *interesting* — that needs a comparison, which is Module 04.

Next: **01a — What a corpus can and cannot answer**.